<a href="https://colab.research.google.com/github/liliquin0990-byte/TestRepo/blob/main/practica_sql_ventas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sqlite3
import pandas as pd

# 1. Creamos una base de datos en memoria
conexion = sqlite3.connect(':memory:')

# 2. Creamos la tabla de ventas del comercio electrónico
datos = [
    (101, '2026-11-01', 'Electrónica', 'Auriculares', 2, 25.0),
    (102, '2026-11-01', 'Hogar', 'Lámpara LED', 1, 30.0),
    (103, '2026-11-02', 'Electrónica', 'Teclado', 1, 45.0),
    (104, '2026-11-02', 'Electrónica', 'Mouse', 3, 15.0),
    (105, '2026-11-03', 'Hogar', 'Cafetera', 1, 60.0),
    (106, '2026-11-03', 'Electrónica', 'Auriculares', 1, 25.0)
]

tabla = pd.DataFrame(datos, columns=['ID_Venta', 'Fecha', 'Categoria', 'Producto', 'Cantidad', 'Precio'])
tabla.to_sql('ventas', conexion, index=False)

print("Base de datos 'ventas' lista para consultar con SQL.")


Base de datos 'ventas' lista para consultar con SQL.


In [3]:
# Escribimos nuestra consulta en lenguaje SQL
consulta = """
SELECT ID_Venta, Producto, Cantidad, Precio
FROM ventas
WHERE Categoria = 'Electrónica'
"""

# Ejecutamos la consulta y mostramos el resultado en una tabla
resultado = pd.read_sql_query(consulta, conexion)
print(resultado)


   ID_Venta     Producto  Cantidad  Precio
0       101  Auriculares         2    25.0
1       103      Teclado         1    45.0
2       104        Mouse         3    15.0
3       106  Auriculares         1    25.0


In [4]:
consulta_ingresos = """
SELECT
    Categoria,
    SUM(Cantidad) AS Unidades_Vendidas,
    SUM(Cantidad * Precio) AS Total_Ingresos
FROM ventas
GROUP BY Categoria
ORDER BY Total_Ingresos DESC
"""

resumen_sql = pd.read_sql_query(consulta_ingresos, conexion)
print(resumen_sql)


     Categoria  Unidades_Vendidas  Total_Ingresos
0  Electrónica                  7           165.0
1        Hogar                  2            90.0


In [5]:
# 1. Creamos una segunda tabla con los datos de nuestros clientes
datos_clientes = [
    (1, 'Ana Gómez', 'Asunción'),
    (2, 'Carlos Ruiz', 'Encarnación'),
    (3, 'María López', 'Ciudad del Este')
]
tabla_clientes = pd.DataFrame(datos_clientes, columns=['ID_Cliente', 'Nombre', 'Ciudad'])
tabla_clientes.to_sql('clientes', conexion, index=False, if_exists='replace')

# 2. Actualizamos nuestras ventas agregando qué ID_Cliente hizo cada compra
datos_ventas_con_cliente = [
    (101, 1, 'Electrónica', 'Auriculares', 2, 25.0),
    (102, 2, 'Hogar', 'Lámpara LED', 1, 30.0),
    (103, 1, 'Electrónica', 'Teclado', 1, 45.0),
    (104, 3, 'Electrónica', 'Mouse', 3, 15.0),
    (105, 2, 'Hogar', 'Cafetera', 1, 60.0),
    (106, 1, 'Electrónica', 'Auriculares', 1, 25.0)
]
tabla_ventas2 = pd.DataFrame(datos_ventas_con_cliente, columns=['ID_Venta', 'ID_Cliente', 'Categoria', 'Producto', 'Cantidad', 'Precio'])
tabla_ventas2.to_sql('ventas_clientes', conexion, index=False, if_exists='replace')

# 3. Usamos JOIN para unir ambas tablas y calcular cuánto gastó cada cliente
consulta_join = """
SELECT
    c.Nombre,
    c.Ciudad,
    SUM(v.Cantidad * v.Precio) AS Total_Gastado
FROM ventas_clientes v
JOIN clientes c ON v.ID_Cliente = c.ID_Cliente
GROUP BY c.Nombre, c.Ciudad
ORDER BY Total_Gastado DESC
"""

resultado_clientes = pd.read_sql_query(consulta_join, conexion)
print(resultado_clientes)


        Nombre           Ciudad  Total_Gastado
0    Ana Gómez         Asunción          120.0
1  Carlos Ruiz      Encarnación           90.0
2  María López  Ciudad del Este           45.0
